# Trial 3: full-sequence 180 cm adaptation

A continuous target-body animation for all **503 native frames (0–502)**. This companion notebook reads the existing Trial 3 caches and saves a separate diagnostic run. The original analysis notebook and single-move results are preserved.

The fit uses one joint optimization over the entire sequence, fixed body lengths, variable contacts, and temporal smoothness. Red and uncertain frames remain visible with numerical residuals. This is not a physically validated climb or a success prediction.

Run from the repository with its `.venv` kernel. Restart the kernel after editing helper modules. See [method and limitations](docs/trial3_full_sequence.md).

## 1. Target body and verified inputs


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Video, display
from beta_translator.inverse_kinematics import RATIOS
from beta_translator.full_sequence_pipeline import run_full_sequence

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'beta_translator').is_dir():
    raise FileNotFoundError('Open this notebook from the Beta_Translator repository.')
TARGET_HEIGHT_CM = 180
TARGET_RATIOS = dict(RATIOS)
print('Target body:', TARGET_HEIGHT_CM, 'cm |', TARGET_RATIOS)
print('Source climber height: unknown')


Target body: 180 cm | {'upper_arm': 0.186, 'forearm': 0.146, 'thigh': 0.245, 'shank': 0.246, 'shoulder_w': 0.259, 'hip_w': 0.191, 'torso': 0.288}
Source climber height: unknown


## 2. Fit all frames together

Existing pose, contact, map, and calibration caches are verified before use. No model training or hold-map reconstruction occurs. Each hand and foot can change state independently. The solver exports a finite best-effort trajectory with diagnostic flags, including unmet contacts.

H053 remains an explicitly flagged extrapolation; its calibration coverage has not been extended. A `free` foot is labeled as unresolved ground/wall/air support. Output directories are content-addressed, and earlier runs are preserved.


In [2]:
full_sequence = run_full_sequence(
    PROJECT_DIR, height_cm=TARGET_HEIGHT_CM, ratios=TARGET_RATIOS,
    progress=lambda message: print(message, flush=True),
)
RUN_DIR = Path(full_sequence['output_dir'])
print('Saved run:', RUN_DIR)
print('Optimizer:', full_sequence['optimizer'])


Reused verified full-sequence outputs.
Saved run: /Users/ziwei/Desktop/2026 Fall/CIS 5810/Beta_Translator/outputs/trial3/full_sequence/target_180cm_250e15b3ee81
Optimizer: {'success': True, 'message': '`ftol` termination condition is satisfied.', 'nfev': 590, 'cost': 2612.4341293470875, 'optimality': 0.9408920306302944}


## 3. Full-duration target-body video

The video has one output frame per analysis frame, at the native 30 FPS (about 16.77 seconds). It shows the fitted 180 cm skeleton on the reconstructed wall. Dashed lines show missed targets; the side panel and timeline expose uncertainty and constraint violations.

Green means a numerical contact gap at most 0.1 cm, not real-world accuracy. Red means a numerical mismatch. Purple marks geometry outside observed-hole coverage. Amber indicates free or uncertain contacts. The target body has fixed segment lengths throughout.


In [3]:
display(Video(url=(RUN_DIR / 'trial3_target_180cm_full_sequence.mp4').relative_to(PROJECT_DIR).as_posix(), width=900))
summary_keys = ['frame_count', 'first_frame', 'last_frame', 'fps', 'duration_seconds',
                'max_contact_error_cm', 'max_segment_length_error_cm',
                'max_angle_step_deg', 'max_hip_step_cm']
display(pd.DataFrame([{'metric':k, 'value':full_sequence[k]} for k in summary_keys]))
display(pd.DataFrame([{'issue':k, 'frames':v} for k,v in full_sequence['flag_counts'].items()]))


,metric,value
0,frame_count,5.030000e+02
1,first_frame,0.000000e+00
2,last_frame,5.020000e+02
3,fps,3.000000e+01
4,duration_seconds,1.676667e+01
5,max_contact_error_cm,1.481282e+01
6,max_segment_length_error_cm,3.552714e-14
7,max_angle_step_deg,1.626742e+01
8,max_hip_step_cm,6.014432e+00


,issue,frames
0,contact_residual,245
1,large_contact_residual,31
2,span_conflict,83
3,hold_extrapolation,93
4,unknown_contact,0
5,unresolved_foot_support,168
6,wall_contact_unlocated,0
7,low_contact_confidence,276
8,low_pose_visibility,503
9,reference_extrapolation,277


## 4. Inspect problem frames

Counts can overlap. A frame may simultaneously have an infeasible contact span, an uncertain prediction, and an occluded reference joint. The CSV files retain the original decoded labels and per-limb gaps. `trajectory.npz` stores all 503 fitted poses, references, targets, and masks.

No ground contact location, force, friction, balance, grasp shape, collision, or body/hold depth is inferred. The animation does not automatically replan an alternative beta.


In [4]:
frame_diagnostics = pd.read_csv(RUN_DIR / 'frame_diagnostics.csv')
limb_diagnostics = pd.read_csv(RUN_DIR / 'limb_diagnostics.csv')
issue_intervals = pd.read_csv(RUN_DIR / 'issue_intervals.csv')
display(frame_diagnostics.nlargest(12, 'max_contact_error_cm')[
    ['frame', 'time_sec', 'max_contact_error_cm', 'span_excess_cm', 'issues']])
display(issue_intervals[issue_intervals['issue'].isin(
    ['span_conflict', 'hold_extrapolation', 'large_contact_residual', 'rapid_pose_change'])])
print('Full frame table:', RUN_DIR / 'frame_diagnostics.csv')
print('Per-limb table:', RUN_DIR / 'limb_diagnostics.csv')
print('Provenance and settings:', RUN_DIR / 'summary.json')


Full frame table: /Users/ziwei/Desktop/2026 Fall/CIS 5810/Beta_Translator/outputs/trial3/full_sequence/target_180cm_250e15b3ee81/frame_diagnostics.csv
Per-limb table: /Users/ziwei/Desktop/2026 Fall/CIS 5810/Beta_Translator/outputs/trial3/full_sequence/target_180cm_250e15b3ee81/limb_diagnostics.csv
Provenance and settings: /Users/ziwei/Desktop/2026 Fall/CIS 5810/Beta_Translator/outputs/trial3/full_sequence/target_180cm_250e15b3ee81/summary.json


,frame,time_sec,max_contact_error_cm,span_excess_cm,issues
486,486,16.200000,14.812816,14.169135,contact_residual;large_contact_residual;span_c...
256,256,8.533333,12.029357,16.725864,contact_residual;large_contact_residual;span_c...
493,493,16.433333,11.041590,14.169135,contact_residual;large_contact_residual;span_c...
487,487,16.233333,11.009625,14.169135,contact_residual;large_contact_residual;span_c...
126,126,4.200000,10.859301,0.000000,contact_residual;large_contact_residual;unreso...
494,494,16.466667,10.800861,14.169135,contact_residual;large_contact_residual;span_c...
257,257,8.566667,10.610425,16.725864,contact_residual;large_contact_residual;span_c...
492,492,16.400000,10.286842,14.169135,contact_residual;large_contact_residual;span_c...
501,501,16.700000,10.267785,14.169135,contact_residual;large_contact_residual;span_c...
502,502,16.733333,10.230983,14.169135,contact_residual;large_contact_residual;span_c...


,issue,start_frame,end_frame,start_sec,end_exclusive_sec
0,hold_extrapolation,0,92,0.000000,3.100000
22,large_contact_residual,125,126,4.166667,4.233333
23,rapid_pose_change,126,127,4.200000,4.266667
36,large_contact_residual,256,263,8.533333,8.800000
37,span_conflict,256,263,8.533333,8.800000
50,large_contact_residual,354,355,11.800000,11.866667
54,rapid_pose_change,372,372,12.400000,12.433333
56,large_contact_residual,378,379,12.600000,12.666667
57,rapid_pose_change,379,379,12.633333,12.666667
63,span_conflict,428,502,14.266667,16.766667
